# Wikipedia text embeddings

This notebook builds a text representation of each fighter from their Wikipedia article and tests one use of it: whether cosine similarity between those representations produces a meaningful "similar fighters" readout to sit alongside the two axes.

The pipeline verifies that each resolved article is a genuine fighter biography, extracts the lead section, and embeds it as a sentence vector (all-MiniLM-L6-v2, and then the stronger all-mpnet-base-v2). Whether the similarity that follows is meaningful is treated as a question rather than an assumption, so the nearest neighbours are assessed by structured inspection of known fighters. The outcome of that inspection is recorded at the close of the notebook.

In [ ]:
# BLOCK 1: library imports, install, and data location

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import requests, time, re, unicodedata
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

# data location (portable across Drive and a local repo checkout)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    pass  # not in Colab

DRIVE_DIR = Path('/content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/'
                 'Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs')
OUTPUT_DIR = DRIVE_DIR if DRIVE_DIR.exists() else Path('./data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using data directory: {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Section 1: Verifying article resolution

Each fighter was linked to a Wikipedia article title when the pageview data was gathered, but a resolved title is not always the fighter's biography. Some titles link to event or list pages rather than the biography needed here. This block checks each resolved article and flags the ones that are not genuine biographies, so a non biography cannot feed the embedding step.

A biography is identified by its opening: the check looks for the fighter's name, a date of birth, and an identification of the subject as a mixed martial artist.

In [ ]:
# BLOCK 2: verify each resolved article is a fighter biography

tm = pd.read_parquet(OUTPUT_DIR / 'pageview_title_map.parquet')
active = pd.read_parquet(OUTPUT_DIR / 'active_universe.parquet')
TITLE = 'resolved_title' if 'resolved_title' in tm else tm.columns[1]

resolved = tm.merge(active[['FIGHTER']], on='FIGHTER', how='inner')
resolved = resolved[resolved[TITLE].notna()].copy()
print(f'{len(resolved)} active fighters with a resolved title')

SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'UFC-MSc-project/1.0 (academic research; contact via GitHub th1555)'})

def strip_accents(s):
    # fold accented characters to plain ascii so "procházka" matches "prochazka"
    return ''.join(c for c in unicodedata.normalize('NFKD', s)
                   if not unicodedata.combining(c)).lower()

def fetch_intro(title):
    for attempt in range(4):
        try:
            r = SESSION.get('https://en.wikipedia.org/w/api.php', params={
                'action': 'query', 'prop': 'extracts', 'explaintext': 1,
                'exintro': 1, 'titles': title, 'format': 'json', 'redirects': 1,
            }, timeout=20)
            if r.status_code == 200:
                pages = r.json().get('query', {}).get('pages', {})
                if pages:
                    return next(iter(pages.values())).get('extract', ''), 'ok'
            elif r.status_code == 429:
                time.sleep(5 * (attempt + 1)); continue
        except Exception:
            time.sleep(2 * (attempt + 1))
    return '', 'failed'

BAD_TITLE = re.compile(r'^UFC \d|^\d{4} in |List of|UFC rankings| in UFC$| in Oktagon', re.I)

rows = []
for i, (_, r) in enumerate(resolved.iterrows(), 1):
    title = r[TITLE]
    if BAD_TITLE.search(title):
        rows.append({'FIGHTER': r['FIGHTER'], 'resolved_title': title,
                     'intro_words': 0, 'verdict': 'bad_title_pattern'})
        continue
    text, status = fetch_intro(title)
    if status == 'failed':
        verdict = 'fetch_failed'
    elif len(text.split()) < 20:
        verdict = 'too_short'
    else:
        head = strip_accents(text[:400])
        # check every name part, not just surname; accents folded on both sides.
        # a correct article names the fighter; require the surname, or most name parts, present
        parts = [strip_accents(p) for p in r['FIGHTER'].split() if len(p) > 1]
        surname = parts[-1] if parts else ''
        hits = sum(p in head for p in parts)
        verdict = 'ok' if (surname in head or hits >= max(1, len(parts) - 1)) else 'name_mismatch'
    rows.append({'FIGHTER': r['FIGHTER'], 'resolved_title': title,
                 'intro_words': len(text.split()), 'verdict': verdict})
    if i % 100 == 0:
        print(f'  {i}/{len(resolved)}')
    time.sleep(0.5)

check = pd.DataFrame(rows)
print('\nverdict counts:')
print(check['verdict'].value_counts().to_string())
print('\nname_mismatch cases (review these):')
print(check[check['verdict']=='name_mismatch'][['FIGHTER','resolved_title']].to_string(index=False))
check.to_parquet(OUTPUT_DIR / 'article_bio_check.parquet', index=False)

486 active fighters with a resolved title
  100/486
  200/486
  300/486
  400/486

verdict counts:
verdict
ok              443
fetch_failed     43

name_mismatch cases (review these):
Empty DataFrame
Columns: [FIGHTER, resolved_title]
Index: []


## Section 2: Lead text extraction

Only fighters whose resolved article passed the biography verification are embedded, so a stub or a wrong article cannot produce a meaningless neighbour list. The unit embedded is the article lead rather than the full text: the lead carries the biographical summary (nationality, era, background, notable achievements), it sits within the embedding model's token window, and the fight-by-fight sections that fill the rest of an article are results, which the competitive axis already captures and which would add noise to a similarity comparison.

In [ ]:
# BLOCK 3: lead section text for each verified biography

LEADS_PATH = OUTPUT_DIR / 'fighter_leads.parquet'

if LEADS_PATH.exists():
    bios = pd.read_parquet(LEADS_PATH)
    print(f'reloaded {len(bios)} leads from disk (no fetch)')
else:
    check = pd.read_parquet(OUTPUT_DIR / 'article_bio_check.parquet')
    bios = check[check['verdict'] == 'ok'].copy()
    print(f'{len(bios)} verified biographies to fetch')

    def fetch_lead(title):
        for attempt in range(4):
            try:
                r = SESSION.get('https://en.wikipedia.org/w/api.php', params={
                    'action': 'query', 'prop': 'extracts', 'explaintext': 1,
                    'exintro': 1, 'titles': title, 'format': 'json', 'redirects': 1,
                }, timeout=20)
                if r.status_code == 200:
                    pages = r.json().get('query', {}).get('pages', {})
                    if pages:
                        return next(iter(pages.values())).get('extract', '')
                elif r.status_code == 429:
                    time.sleep(5 * (attempt + 1)); continue
            except Exception:
                time.sleep(2 * (attempt + 1))
        return ''

    texts = []
    for i, (_, r) in enumerate(bios.iterrows(), 1):
        texts.append(fetch_lead(r['resolved_title']))
        if i % 100 == 0:
            print(f'  {i}/{len(bios)}')
        time.sleep(0.5)
    bios['lead_text'] = texts
    bios['lead_words'] = bios['lead_text'].str.split().str.len()
    bios = bios[bios['lead_words'] >= 20].reset_index(drop=True)
    bios.to_parquet(LEADS_PATH, index=False)
    print(f'\nsaved {len(bios)} leads to {LEADS_PATH.name}')

443 verified biographies to fetch
  100/443
  200/443
  300/443
  400/443

saved 403 leads to fighter_leads.parquet


## Section 3: Embedding and nearest neighbours

Each lead is mapped to a 384-dimensional vector with all-MiniLM-L6-v2, a general purpose sentence embedding model. The model is not trained on mixed martial arts, so the similarity it produces reflects biographical text (background, era, nationality, career shape) rather than fight style; this is stated as the bound on the component rather than claimed as stylistic understanding.

The same leads are also embedded with the stronger all-mpnet-base-v2, so the two models can be compared on the same neighbour inspection.

In [ ]:
# BLOCK 4: embed with two models and compare the nearest neighbours
from sentence_transformers import SentenceTransformer
import numpy as np

EYEBALL = ['Islam Makhachev', 'Tony Ferguson', 'Alexander Volkanovski',
           'Shavkat Rakhmonov', 'Jon Jones', 'Zhang Weili']
K = 5

def embed_and_neighbours(model_name, tag):
    # embed the leads, take cosine nearest neighbours, save both artefacts under the model tag
    model = SentenceTransformer(model_name)
    emb = model.encode(bios['lead_text'].tolist(), batch_size=32,
                       show_progress_bar=True, normalize_embeddings=True)
    emb_df = pd.DataFrame(emb, index=bios['FIGHTER'])
    emb_df.to_parquet(OUTPUT_DIR / f'fighter_embeddings_{tag}.parquet')

    names = emb_df.index.to_numpy()
    mat = emb_df.to_numpy()
    sim = mat @ mat.T
    np.fill_diagonal(sim, -1.0)

    rows = []
    for i, name in enumerate(names):
        for rank, j in enumerate(np.argsort(sim[i])[::-1][:K], 1):
            rows.append({'FIGHTER': name, 'rank': rank,
                         'similar_to': names[j], 'similarity': round(float(sim[i, j]), 3)})
    nb = pd.DataFrame(rows)
    nb.to_parquet(OUTPUT_DIR / f'fighter_similarity_{tag}.parquet', index=False)
    print(f'{tag}: embedded {emb.shape[0]} fighters, dim {emb.shape[1]}')
    return nb

def show(nb, heading):
    print(f'\n Close matches s: {heading}')
    present = set(nb['FIGHTER'])
    for who in EYEBALL:
        if who not in present:
            print(f'\n{who}: not in the embedded set'); continue
        print(f'\n{who}:')
        for _, r in nb[nb['FIGHTER'] == who].head(K).iterrows():
            print(f"  {r['similarity']:.3f}  {r['similar_to']}")

nb_mini = embed_and_neighbours('all-MiniLM-L6-v2', 'minilm')
nb_mpnet = embed_and_neighbours('all-mpnet-base-v2', 'mpnet')

show(nb_mini, 'all-MiniLM-L6-v2')
show(nb_mpnet, 'all-mpnet-base-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

minilm: embedded 403 fighters, dim 384


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

mpnet: embedded 403 fighters, dim 768

 Close matches s: all-MiniLM-L6-v2

Islam Makhachev:
  0.711  Merab Dvalishvili
  0.690  Shavkat Rakhmonov
  0.688  Yaroslav Amosov
  0.674  Alexander Volkanovski
  0.666  Sergei Pavlovich

Tony Ferguson:
  0.710  Anthony Hernandez
  0.680  Tallison Teixeira
  0.670  Alex Perez
  0.668  Michael Morales
  0.660  Alex Pereira

Alexander Volkanovski:
  0.746  Max Holloway
  0.712  Conor McGregor
  0.674  Islam Makhachev
  0.672  Shavkat Rakhmonov
  0.662  Arman Tsarukyan

Shavkat Rakhmonov:
  0.712  Nassourdine Imavov
  0.703  Said Nurmagomedov
  0.690  Islam Makhachev
  0.684  Arman Tsarukyan
  0.684  Aleksandar Rakic

Jon Jones:
  0.668  Justin Gaethje
  0.662  Chris Weidman
  0.662  Conor McGregor
  0.654  Aljamain Sterling
  0.639  Henry Cejudo

Zhang Weili:
  0.761  Yan Xiaonan
  0.752  Xiong Jingnan
  0.729  Zhang Mingyang
  0.679  Tatiana Suarez
  0.643  Valentina Shevchenko

 Close matches s: all-mpnet-base-v2

Islam Makhachev:
  0.740  Shavk

## Section 4: Outcome (negative result)

The similarity captures biographical text resemblance rather than competitive or stylistic resemblance. It recovers coherent groupings for fighters whose coverage is dominated by distinctive national and divisional markers (for example Zhang Weili and the Chinese contingent, or the Caucasus and Central Asian grapplers around Makhachev and Rakhmonov), but returns confident false matches across weight classes (the lightweight Tony Ferguson is nearest to middleweights such as Anthony Hernandez and Gregory Rodrigues) and one driven by a shared surname (Jon Jones nearest to Mason Jones). Moving from all-MiniLM-L6-v2 to the stronger all-mpnet-base-v2 raised the confidence of these errors rather than correcting them, which indicates the discriminating signal is absent from the source text.

The component is therefore retained as an exploratory result and excluded from the live framework; nothing downstream consumes the embedding or similarity outputs. A domain adapted embedding, or a hybrid that constrains textual similarity by structured attributes, is recorded as future work (deviation log C.18).